# 03 Block-Level Causal Discovery

This notebook performs block-level causal discovery over the nine semantic biomedical domains.

Pipeline:
Domain scores → Background Knowledge → PC/FCI → validation → transitive reduction → visualization/export.

In [ ]:
import sys
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import GraphicalLassoCV
import numpy as np
import pandas as pd
import networkx as nx

PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))


from utils.background_knowledge import (
    build_expert_background_knowledge,
    build_hybrid_background_knowledge,
    EXPERT_DOMAIN_EDGES,
    print_expert_dag,
)
from utils.causal_discovery import prepare_causal_data, run_pc_discovery, run_fci_discovery
from utils.graph_utils import (
    check_cycles,
    check_endpoint_violations,
    transitive_reduce_edges,
    graph_statistics,
    print_edges,
    save_edges,
)
from utils.visualization import (
    visualize_causal_dag,
    plot_precision_heatmap,
    visualize_undirected_skeleton

)
print("Libraries loaded.")

In [ ]:
# Folder containing the full-cohort notebooks
EXPERIMENT_DIR = Path.cwd().resolve()

# Main project directory
PROJECT_ROOT = EXPERIMENT_DIR.parents[1]

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = EXPERIMENT_DIR / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"
GRAPHS_DIR = RESULTS_DIR / "graphs"

for folder in [FIGURES_DIR, TABLES_DIR, GRAPHS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Paths ready.")

In [ ]:
df_blocks = pd.read_csv(PROCESSED_DIR / "attica_domain_scores_full.csv")

blocks_to_use = [
    "Demographics",
    "SES",
    "Behavioral",
    "Treatment",
    "Obesity",
    "Metabolic",
    "Biomarkers",
    "Conditions",
    "Outcomes",
]

df_blocks = df_blocks[blocks_to_use].copy()

print("Domain score dataset:", df_blocks.shape)
display(df_blocks.head())

In [ ]:
df_blocks_cc, data_blocks, block_names = prepare_causal_data(df_blocks)

print("Complete-case shape:", df_blocks_cc.shape)
print("Blocks:", block_names)
print("Any NaNs:", np.isnan(data_blocks).any())

In [ ]:
print("Missing values:")
display(df_blocks.isna().sum())

print("\nUnique values:")
display(df_blocks.nunique())

print("\nSummary:")
display(df_blocks.describe())

print("\nCorrelation matrix:")
display(df_blocks.corr())

In [ ]:
# ============================================================
# GRAPHICAL LASSO SPARSE DEPENDENCY ESTIMATION
# ============================================================

print("=" * 80)
print("GRAPHICAL LASSO SPARSE SKELETON")
print("=" * 80)

X = df_blocks[blocks_to_use].copy()

# Median imputation
imputer = SimpleImputer(strategy="median")

X_imputed = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns,
    index=X.index,
)

# Standardisation
scaler = StandardScaler()

X_scaled = pd.DataFrame(
    scaler.fit_transform(X_imputed),
    columns=X.columns,
    index=X.index,
)

# Cross-validated sparse precision estimation
glasso = GraphicalLassoCV(cv=5)
glasso.fit(X_scaled)

precision = pd.DataFrame(
    glasso.precision_,
    index=blocks_to_use,
    columns=blocks_to_use,
)

# Convert precision matrix to partial correlations
precision_array = precision.to_numpy()

denominator = np.sqrt(
    np.outer(
        np.diag(precision_array),
        np.diag(precision_array),
    )
)

partial_corr_array = (
    -precision_array / denominator
)

np.fill_diagonal(partial_corr_array, 1.0)

partial_corr = pd.DataFrame(
    partial_corr_array,
    index=blocks_to_use,
    columns=blocks_to_use,
)

print(f"Selected Graphical Lasso alpha: {glasso.alpha_:.6f}")

display(partial_corr.round(3))

In [ ]:
# ============================================================
# GRAPHICAL LASSO CANDIDATE TABLE
# ============================================================

candidate_records = []

for i in range(len(blocks_to_use)):
    for j in range(i + 1, len(blocks_to_use)):

        node_1 = blocks_to_use[i]
        node_2 = blocks_to_use[j]

        precision_value = float(precision.loc[node_1, node_2])
        partial_value = float(partial_corr.loc[node_1, node_2])

        # GraphicalLassoCV selects the regularisation strength automatically.
        # The sparse graph is defined directly by non-zero off-diagonal
        # precision entries; no additional partial-correlation threshold is used.
        retained = precision_value != 0.0

        candidate_records.append({
            "Node1": node_1,
            "Node2": node_2,
            "PrecisionEntry": precision_value,
            "PartialCorr": partial_value,
            "AbsPartialCorr": abs(partial_value),
            "RetainedByGraphicalLasso": retained,
        })

candidate_df = (
    pd.DataFrame(candidate_records)
    .sort_values(
        "AbsPartialCorr",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(candidate_df)


In [ ]:
# ============================================================
# GRAPHICAL-LASSO-SUPPORTED UNDIRECTED SKELETON
# ============================================================

lasso_pairs = {
    tuple(sorted((
        row["Node1"],
        row["Node2"],
    )))
    for _, row in candidate_df.iterrows()
    if row["RetainedByGraphicalLasso"]
}

sparse_edges = [
    (
        row["Node1"],
        row["Node2"],
        row["PartialCorr"],
    )
    for _, row in candidate_df.iterrows()
    if row["RetainedByGraphicalLasso"]
]

print("=" * 80)
print("GRAPHICAL-LASSO-SUPPORTED PAIRS")
print("=" * 80)

print(f"Cross-validated Graphical Lasso alpha: {glasso.alpha_:.6f}")
print("Selection rule: non-zero off-diagonal precision entries")
print(f"Number of sparse pairs: {len(lasso_pairs)}")

for pair in sorted(lasso_pairs):
    print(pair)

display(
    pd.DataFrame(
        sparse_edges,
        columns=[
            "Node1",
            "Node2",
            "PartialCorrelation",
        ],
    )
)


In [ ]:
visualize_undirected_skeleton(
    edges=sparse_edges,
    block_names=block_names,
    title="Sparse Dependency Skeleton (Graphical Lasso)",
    filename=FIGURES_DIR / "graphical_lasso_skeleton.png"
)


In [ ]:
# ============================================================
# ZERO-CENTRED PARTIAL-CORRELATION HEATMAP
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# ------------------------------------------------------------
# Convert precision matrix safely to a NumPy array
# ------------------------------------------------------------

if isinstance(precision, pd.DataFrame):
    precision_array = precision.to_numpy(dtype=float)

    # Preserve names from the DataFrame when available
    matrix_names = list(precision.index)

else:
    precision_array = np.asarray(
        precision,
        dtype=float,
    )

    matrix_names = list(block_names)

if precision_array.ndim != 2:
    raise ValueError(
        "The precision matrix must be two-dimensional."
    )

if (
    precision_array.shape[0]
    != precision_array.shape[1]
):
    raise ValueError(
        "The precision matrix must be square."
    )

if len(matrix_names) != precision_array.shape[0]:
    raise ValueError(
        "The number of block names does not match "
        "the precision-matrix dimensions."
    )

# ------------------------------------------------------------
# Convert precision matrix to partial correlations
# ------------------------------------------------------------

precision_diagonal = np.diag(
    precision_array
)

if np.any(precision_diagonal <= 0):
    raise ValueError(
        "The precision-matrix diagonal contains "
        "non-positive values."
    )

denominator = np.sqrt(
    np.outer(
        precision_diagonal,
        precision_diagonal,
    )
)

partial_correlation_array = (
    -precision_array / denominator
)

# The diagonal represents self-correlation and is not informative
np.fill_diagonal(
    partial_correlation_array,
    0.0,
)

partial_correlation_df = pd.DataFrame(
    partial_correlation_array,
    index=matrix_names,
    columns=matrix_names,
)

# ------------------------------------------------------------
# Create symmetric, zero-centred colour scale
# ------------------------------------------------------------

off_diagonal_mask = ~np.eye(
    len(matrix_names),
    dtype=bool,
)

colour_limit = float(
    np.nanmax(
        np.abs(
            partial_correlation_array[
                off_diagonal_mask
            ]
        )
    )
)

if not np.isfinite(colour_limit) or colour_limit == 0:
    colour_limit = 1.0

norm = TwoSlopeNorm(
    vmin=-colour_limit,
    vcenter=0.0,
    vmax=colour_limit,
)

# ------------------------------------------------------------
# Plot heatmap
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(11, 9)
)

image = ax.imshow(
    partial_correlation_array,
    cmap="RdBu_r",
    norm=norm,
    aspect="equal",
)

ax.set_xticks(
    range(len(matrix_names))
)

ax.set_xticklabels(
    matrix_names,
    rotation=45,
    ha="right",
)

ax.set_yticks(
    range(len(matrix_names))
)

ax.set_yticklabels(
    matrix_names
)

for row in range(len(matrix_names)):
    for column in range(len(matrix_names)):

        value = partial_correlation_array[
            row,
            column,
        ]

        ax.text(
            column,
            row,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=8,
        )

colour_bar = fig.colorbar(
    image,
    ax=ax,
    shrink=0.85,
)

colour_bar.set_label(
    "Partial correlation"
)

ax.set_title(
    "Graphical Lasso Partial-Correlation Matrix\n"
    "Zero-centred diverging colour scale"
)

fig.tight_layout()

PARTIAL_CORRELATION_FIGURE = (
    FIGURES_DIR
    / "graphical_lasso_partial_correlation_RdBu.png"
)

fig.savefig(
    PARTIAL_CORRELATION_FIGURE,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

# ------------------------------------------------------------
# Export numerical matrix
# ------------------------------------------------------------

PARTIAL_CORRELATION_TABLE = (
    TABLES_DIR
    / "graphical_lasso_partial_correlations_full.csv"
)

partial_correlation_df.to_csv(
    PARTIAL_CORRELATION_TABLE
)

print("=" * 90)
print("PARTIAL-CORRELATION MATRIX EXPORTED")
print("=" * 90)

print("Figure:")
print(PARTIAL_CORRELATION_FIGURE)

print("\nTable:")
print(PARTIAL_CORRELATION_TABLE)

display(
    partial_correlation_df.round(3)
)

In [ ]:
cg_pc_unconstrained, pc_unconstrained_edges = run_pc_discovery(
    data=data_blocks,
    node_names=block_names,
    background_knowledge=None,
    alpha=0.05,
    stable=True,
    show_progress=True,
)

print_edges(pc_unconstrained_edges, "PC directed edges")

In [ ]:
visualize_causal_dag(
    edges=pc_unconstrained_edges,
    block_names=block_names,
    title="PC Algorithm without BK- Block-Level Causal Graph",
    filename=FIGURES_DIR / "pc_withoutbk_block_dag.png"
)


In [ ]:
G_pc_unconstrained = check_cycles(
    pc_unconstrained_edges,
    graph_name="Unconstrained PC directed projection",
    node_names=block_names,
)

display(
    graph_statistics(
        pc_unconstrained_edges,
        graph_name="Unconstrained PC",
        node_names=block_names,
    )
)

pc_unconstrained_edges_reduced = transitive_reduce_edges(
    pc_unconstrained_edges,
    graph_name="Unconstrained PC",
    node_names=block_names,
)

print_edges(
    pc_unconstrained_edges_reduced,
    "Unconstrained PC transitive-reduced edges",
)

In [ ]:
fci_unconstrained_graph, fci_unconstrained_edges = run_fci_discovery(
    data=data_blocks,
    node_names=block_names,
    background_knowledge=None,
    alpha=0.05,
    depth=-1,
    max_path_length=-1,
    show_progress=True,
)

In [ ]:
visualize_causal_dag(
    edges=fci_unconstrained_edges,
    block_names=block_names,
    title="Unconstrained FCI: Directed Projection",
    filename=FIGURES_DIR / "fci_withoutbk_block_dag.png"
)

In [ ]:
G_fci_unconstrained = check_cycles(
    fci_unconstrained_edges,
    graph_name="Unconstrained FCI directed projection",
    node_names=block_names,
)

display(
    graph_statistics(
        fci_unconstrained_edges,
        graph_name="Unconstrained FCI",
        node_names=block_names,
    )
)

fci_unconstrained_edges_reduced = transitive_reduce_edges(
    fci_unconstrained_edges,
    graph_name="Unconstrained FCI",
    node_names=block_names,
)

print_edges(
    fci_unconstrained_edges_reduced,
    "Unconstrained FCI transitive-reduced edges",
)

In [ ]:
bk_expert, expert_nodes, expert_edges_from_bk = build_expert_background_knowledge(
    blocks_to_use=blocks_to_use,
    expert_edges=EXPERT_DOMAIN_EDGES,
)

print_expert_dag(expert_edges_from_bk)

# Visualize the expert DAG itself because it is the single prior
# structural model used to derive directional background knowledge.
visualize_causal_dag(
    edges=expert_edges_from_bk,
    block_names=block_names,
    title="Expert Cardiovascular DAG - Prior Structural Knowledge (Full Cohort)",
    filename=FIGURES_DIR / "expert_domain_dag_full_cohort.png",
)


In [ ]:
cg_pc_expert, pc_expert_edges = run_pc_discovery(
    data=data_blocks,
    node_names=block_names,
    background_knowledge=bk_expert,
    alpha=0.05,
    stable=True,
    show_progress=True,
)

print_edges(pc_expert_edges, "PC directed edges")

In [ ]:
check_endpoint_violations(pc_expert_edges)

G_pc = check_cycles(
    pc_expert_edges,
    graph_name="PC with Expert-Derived Background Knowledge",
    node_names=block_names
)

display(graph_statistics(
    pc_expert_edges,
    graph_name="PC with Expert-Derived Background Knowledge",
    node_names=block_names
))

In [ ]:
pc_expert_edges_reduced = transitive_reduce_edges(
    pc_expert_edges,
    graph_name="PC with Expert-Derived Background Knowledge",
    node_names=block_names
)

print_edges(pc_expert_edges_reduced , "PC transitive-reduced edges")

In [ ]:
visualize_causal_dag(
    edges=pc_expert_edges,
    block_names=block_names,
    title="PC Expert-Constrained - Block-Level Causal Graph",
    filename=FIGURES_DIR / "pc_expert_block_dag.png"
)

visualize_causal_dag(
    edges=pc_expert_edges_reduced ,
    block_names=block_names,
    title="PC Expert-Constrained - Transitive-Reduced Causal Graph",
    filename=FIGURES_DIR / "pc_expert_block_dag_reduced.png"
)

In [ ]:
fci_graph, fci_expert_edges = run_fci_discovery(
    data=data_blocks,
    node_names=block_names,
    background_knowledge=bk_expert,
    alpha=0.05,
    depth=-1,
    max_path_length=-1,
    show_progress=True,
)

print_edges(fci_expert_edges, "FCI directed edges")

In [ ]:
check_endpoint_violations(fci_expert_edges)

G_fci = check_cycles(
    fci_expert_edges,
    graph_name="FCI with Expert-Derived Background Knowledge",
    node_names=block_names
)

display(graph_statistics(
    fci_expert_edges,
    graph_name="FCI with Expert-Derived Background Knowledge",
    node_names=block_names
))

In [ ]:
fci_expert_edges_reduced = transitive_reduce_edges(
    fci_expert_edges,
    graph_name="FCI with Expert-Derived Background Knowledge",
    node_names=block_names
)

print_edges(fci_expert_edges_reduced , "FCI transitive-reduced edges")

In [ ]:
visualize_causal_dag(
    edges=fci_expert_edges,
    block_names=block_names,
    title="FCI Expert-Constrained - Block-Level Causal Graph",
    filename=FIGURES_DIR / "fci_expert_block_dag.png"
)

visualize_causal_dag(
    edges=fci_expert_edges_reduced,
    block_names=block_names,
    title="FCI Expert-Constrained - Transitive-Reduced Causal Graph",
    filename=FIGURES_DIR / "fci_expert_block_dag_reduced.png"
)

In [ ]:
# ============================================================
# AUTOMATIC CONFOUNDER RECOVERY DIAGNOSTIC
# ============================================================

def has_pair(edges, node_a, node_b):
    """Check adjacency regardless of edge direction."""
    return (
        (node_a, node_b) in edges
        or (node_b, node_a) in edges
    )


diagnostic_rows = []

for method_name, unconstrained_edges, expert_edges in [
    (
        "PC",
        pc_unconstrained_edges,
        pc_expert_edges,
    ),
    (
        "FCI",
        fci_unconstrained_edges,
        fci_expert_edges,
    ),
]:

    treatment_adjacency = has_pair(
        unconstrained_edges,
        "Demographics",
        "Treatment",
    )

    outcome_adjacency = has_pair(
        unconstrained_edges,
        "Demographics",
        "Outcomes",
    )

    oriented_to_treatment = (
        ("Demographics", "Treatment")
        in expert_edges
    )

    oriented_to_outcome = (
        ("Demographics", "Outcomes")
        in expert_edges
    )

    diagnostic_rows.append({
        "Method": method_name,
        "Demographics_Treatment_Data_Adjacency":
            treatment_adjacency,
        "Demographics_Outcomes_Data_Adjacency":
            outcome_adjacency,
        "Expert_Demographics_to_Treatment":
            oriented_to_treatment,
        "Expert_Demographics_to_Outcomes":
            oriented_to_outcome,
        "Recovered_Common_Cause_Candidate": (
            treatment_adjacency
            and outcome_adjacency
            and oriented_to_treatment
            and oriented_to_outcome
        ),
    })


confounder_recovery_df = pd.DataFrame(
    diagnostic_rows
)

display(confounder_recovery_df)

confounder_recovery_df.to_csv(
    TABLES_DIR
    / "automatic_confounder_recovery.csv",
    index=False,
)

if confounder_recovery_df[
    "Recovered_Common_Cause_Candidate"
].all():
    print(
        "\nDemographics was automatically recovered "
        "as adjacent to both Treatment and Outcomes, "
        "then oriented as a common-cause candidate "
        "using expert-derived background knowledge."
    )
else:
    print(
        "\nDemographics was not consistently recovered "
        "across all discovery methods."
    )

In [ ]:
# ============================================================
# HYBRID SPARSE + EXPERT-DAG BACKGROUND KNOWLEDGE
# ============================================================

hybrid_info = build_hybrid_background_knowledge(
    sparse_pairs=lasso_pairs,
    blocks_to_use=blocks_to_use,
    expert_edges=EXPERT_DOMAIN_EDGES,
)

bk_hybrid = hybrid_info["background_knowledge"]
nodes = hybrid_info["nodes"]
expert_pairs = hybrid_info["expert_pairs"]
allowed_pairs = hybrid_info["allowed_pairs"]


print("\n" + "=" * 80)
print("EXPERT DAG PAIRS")
print("=" * 80)

for pair in sorted(expert_pairs):
    print(pair)

print("\n" + "=" * 80)
print("FINAL HYBRID ALLOWED PAIRS")
print("=" * 80)

print(f"Lasso pairs : {len(lasso_pairs)}")
print(f"Expert pairs: {len(expert_pairs)}")
print(f"Allowed union: {len(allowed_pairs)}")

for pair in sorted(allowed_pairs):
    source = (
        "Lasso + Expert"
        if pair in lasso_pairs and pair in expert_pairs
        else "Lasso"
        if pair in lasso_pairs
        else "Expert"
    )

    print(f"{pair}  [{source}]")

In [ ]:
hybrid_skeleton_df = pd.DataFrame(
    [
        {
            "Node1": pair[0],
            "Node2": pair[1],
            "LassoSupported": pair in lasso_pairs,
            "ExpertSupported": pair in expert_pairs,
        }
        for pair in sorted(allowed_pairs)
    ]
)

hybrid_skeleton_df.to_csv(
    TABLES_DIR / "hybrid_candidate_skeleton.csv",
    index=False,
)

display(hybrid_skeleton_df)

In [ ]:
# ============================================================
# HYBRID-RESTRICTED PC
# ============================================================

cg_pc_hybrid, pc_hybrid_edges = run_pc_discovery(
    data=data_blocks,
    node_names=block_names,
    background_knowledge=bk_hybrid,
    alpha=0.05,
    stable=True,
    show_progress=True,
)

print_edges(
    pc_hybrid_edges,
    "Hybrid Sparse + Expert BK: PC directed edges",
)

In [ ]:
check_endpoint_violations(pc_hybrid_edges)

G_pc_hybrid = check_cycles(
    pc_hybrid_edges,
    graph_name="Hybrid Sparse + Expert BK PC",
    node_names=block_names,
)

display(
    graph_statistics(
        pc_hybrid_edges,
        graph_name="Hybrid Sparse + Expert BK PC",
        node_names=block_names,
    )
)

pc_hybrid_edges_reduced = transitive_reduce_edges(
    pc_hybrid_edges,
    graph_name="Hybrid Sparse + Expert BK PC",
    node_names=block_names,
)

print_edges(
    pc_hybrid_edges_reduced,
    "Hybrid PC transitive-reduced edges",
)

In [ ]:
visualize_causal_dag(
    edges=pc_hybrid_edges,
    block_names=block_names,
    title="PC Algorithm: Hybrid Sparse + Expert BK",
    filename=FIGURES_DIR / "pc_hybrid_block_dag.png",
)

visualize_causal_dag(
    edges=pc_hybrid_edges_reduced,
    block_names=block_names,
    title="PC Algorithm: Hybrid Sparse + Expert BK, Reduced",
    filename=FIGURES_DIR / "pc_hybrid_block_dag_reduced.png",
)

In [ ]:
# ============================================================
# HYBRID-RESTRICTED FCI
# ============================================================

fci_hybrid_graph, fci_hybrid_edges = run_fci_discovery(
    data=data_blocks,
    node_names=block_names,
    background_knowledge=bk_hybrid,
    alpha=0.05,
    depth=-1,
    max_path_length=-1,
    show_progress=True,
)

print_edges(
    fci_hybrid_edges,
    "Hybrid Sparse + Expert BK: FCI directed edges",
)

In [ ]:
check_endpoint_violations(fci_hybrid_edges)

G_fci_hybrid = check_cycles(
    fci_hybrid_edges,
    graph_name="Hybrid Sparse + Expert BK FCI",
    node_names=block_names,
)

display(
    graph_statistics(
        fci_hybrid_edges,
        graph_name="Hybrid Sparse + Expert BK FCI",
        node_names=block_names,
    )
)

fci_hybrid_edges_reduced = transitive_reduce_edges(
    fci_hybrid_edges,
    graph_name="Hybrid Sparse + Expert BK FCI",
    node_names=block_names,
)

print_edges(
    fci_hybrid_edges_reduced,
    "Hybrid FCI transitive-reduced edges",
)

In [ ]:
visualize_causal_dag(
    edges=fci_hybrid_edges,
    block_names=block_names,
    title="FCI Algorithm: Hybrid Sparse + Expert BK",
    filename=FIGURES_DIR / "fci_hybrid_block_dag.png",
)

visualize_causal_dag(
    edges=fci_hybrid_edges_reduced,
    block_names=block_names,
    title="FCI Algorithm: Hybrid Sparse + Expert BK, Reduced",
    filename=FIGURES_DIR / "fci_hybrid_block_dag_reduced.png",
)

In [ ]:
def validate_edges_against_allowed_pairs(
    edges,
    allowed_pairs,
    graph_name,
):
    """
    Ensure every discovered directed edge belongs to the
    hybrid candidate skeleton.
    """

    violations = [
        (source, target)
        for source, target in edges
        if tuple(sorted((source, target)))
        not in allowed_pairs
    ]

    print("\n" + "=" * 80)
    print(f"HYBRID SKELETON VALIDATION: {graph_name}")
    print("=" * 80)

    if violations:
        print(
            f"Found {len(violations)} edges outside "
            "the allowed hybrid skeleton:"
        )

        for edge in violations:
            print(edge)

        raise AssertionError(
            f"{graph_name} contains edges outside "
            "the allowed hybrid skeleton."
        )

    print(
        "✓ All discovered directed edges belong to "
        "the allowed hybrid skeleton."
    )

In [ ]:
validate_edges_against_allowed_pairs(
    pc_hybrid_edges,
    allowed_pairs,
    "Hybrid PC",
)

validate_edges_against_allowed_pairs(
    fci_hybrid_edges,
    allowed_pairs,
    "Hybrid FCI",
)

In [ ]:
edge_exports = {
    "pc_unconstrained_edges.csv":
        pc_unconstrained_edges,
    "pc_unconstrained_edges_reduced.csv":
        pc_unconstrained_edges_reduced,

    "fci_unconstrained_edges.csv":
        fci_unconstrained_edges,
    "fci_unconstrained_edges_reduced.csv":
        fci_unconstrained_edges_reduced,

    "pc_expert_edges_full_cohort.csv":
        pc_expert_edges,
    "pc_expert_edges_reduced_full_cohort.csv":
        pc_expert_edges_reduced,

    "fci_expert_edges_full_cohort.csv":
        fci_expert_edges,
    "fci_expert_edges_reduced_full_cohort.csv":
        fci_expert_edges_reduced,

    "pc_hybrid_edges_full_cohort.csv":
        pc_hybrid_edges,
    "pc_hybrid_edges_reduced_full_cohort.csv":
        pc_hybrid_edges_reduced,

    "fci_hybrid_edges_full_cohort.csv":
        fci_hybrid_edges,
    "fci_hybrid_edges_reduced_full_cohort.csv":
        fci_hybrid_edges_reduced,
}

for filename, edges in edge_exports.items():
    save_edges(
        edges,
        GRAPHS_DIR / filename,
    )

print(f"Saved {len(edge_exports)} edge tables.")

In [ ]:
experiment_results = {
    "PC_Unconstrained": {
        "edges": pc_unconstrained_edges,
        "reduced": pc_unconstrained_edges_reduced,
    },
    "FCI_Unconstrained": {
        "edges": fci_unconstrained_edges,
        "reduced": fci_unconstrained_edges_reduced,
    },
    "PC_ExpertConstrained": {
        "edges": pc_expert_edges,
        "reduced": pc_expert_edges_reduced,
    },
    "FCI_ExpertConstrained": {
        "edges": fci_expert_edges,
        "reduced": fci_expert_edges_reduced,
    },
    "PC_Hybrid": {
        "edges": pc_hybrid_edges,
        "reduced": pc_hybrid_edges_reduced,
    },
    "FCI_Hybrid": {
        "edges": fci_hybrid_edges,
        "reduced": fci_hybrid_edges_reduced,
    },
}

summary_records = []

for configuration, result in experiment_results.items():
    graph = check_cycles(
        result["edges"],
        graph_name=configuration,
        node_names=block_names,
    )

    summary_records.append({
        "Configuration": configuration,
        "DirectedEdges": len(result["edges"]),
        "ReducedEdges": len(result["reduced"]),
        "RemovedByReduction": (
            len(result["edges"])
            - len(result["reduced"])
        ),
        "IsDAG": nx.is_directed_acyclic_graph(graph),
    })

block_level_summary = pd.DataFrame(
    summary_records
)

block_level_summary.to_csv(
    TABLES_DIR / "block_level_experiment_summary.csv",
    index=False,
)

display(block_level_summary)